## Imports and Login

In [ ]:
!pip3 install pymupdf
!pip install -U langchain-community
!pip install chromadb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.0/20.0 MB 86.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 86.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 4.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 6.8 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB 98.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 72.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.9/94.9 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 284.2/284.2 kB 25.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 81.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.6/101.6 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.0/16.0 MB 85.5 MB/s eta 0:00:00

In [ ]:
from langchain.chains import RetrievalQA
from langchain.llms import OpenAI
from langchain.embeddings import OpenAIEmbeddings, HuggingFaceEmbeddings
import fitz  # PyMuPDF
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.vectorstores import Chroma
from langchain.llms import HuggingFacePipeline
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from langchain.prompts import PromptTemplate
from langchain.chains import RetrievalQA
from huggingface_hub import login
from google.colab import userdata
import pandas as pd
import numpy as np
import torch
import os

In [ ]:
hf_token = userdata.get('hugging_face_logiin')
login(token=hf_token)

## Vector DB on PDFs Set up

In [ ]:
# Extract text from PDFs
def extract_text_from_pdf(pdf_path):
    doc = fitz.open(pdf_path)
    text = ""
    for page in doc:
        text += page.get_text()
    return text

# Split text into chunks
def split_text(text, chunk_size, overlap):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=overlap
    )
    return text_splitter.split_text(text)

In [ ]:
# Create embeddings and vector store
def create_vector_db(text_chunks, embedding_func, directory):

    embeddings = embedding_func

    vector_store = Chroma.from_texts(
        texts=text_chunks,
        embedding=embeddings,
        persist_directory=directory
    )
    return vector_store

In [ ]:
def create_vec_db(pdf_directory, embedding, chunk_size=5000, chunk_overlap=500, persist_directory="./chroma_db"):

    # Process all PDFs
    all_chunks = []

    # Convert files to chunks
    i = 0
    pdf_dirs = os.listdir(pdf_directory)
    for filename in pdf_dirs:
        print(filename, i, "/", len(pdf_dirs))
        if filename.endswith('.pdf'):
            pdf_path = os.path.join(pdf_directory, filename)
            text = extract_text_from_pdf(pdf_path)
            chunks = split_text(text, chunk_size, chunk_overlap)
            all_chunks.extend(chunks)
        i+=1
    # Create vector database
    return create_vector_db(all_chunks, embedding, persist_directory)

In [ ]:

embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-base-en-v1.5"
    # Alternatives: "BAAI/bge-base-en-v1.5" or "BAAI/bge-large-en-v1.5" for higher quality
)

<ipython-input-7-3ab1857870b0>:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
pdf_directory = "/content/drive/MyDrive/PDFs"
db = create_vec_db(pdf_directory, embeddings, chunk_overlap=150, chunk_size=600)

mayo-ecmo.pdf 0 / 19
cleveland-what-is.pdf 1 / 19
main.pdf 2 / 19
978-3-319-96499-7_Chapter_2.pdf 3 / 19
zrk1158.pdf 4 / 19
rccm.202111-2661CP.pdf 5 / 19
pcc-25-e78.pdf 6 / 19
pcc-25-643.pdf 7 / 19
pcc-24-356.pdf 8 / 19
nihms-1686767.pdf 9 / 19
mat-67-485.pdf 10 / 19
jcm-12-05576.pdf 11 / 19
advances-in-field-of-ecmo.pdf 12 / 19
org0701_0013.pdf 13 / 19
jtd-07-07-E166.pdf 14 / 19
JPC-58-1525.pdf 15 / 19
medicina-61-00493-1.pdf 16 / 19
jcm-10-01573.pdf 17 / 19
predict-survival.pdf 18 / 19


In [ ]:
db

## LLM Set Up

In [ ]:
# model_id = "tiiuae/falcon-7b-instruct"
model_id = "mistralai/Mistral-7B-v0.1"
# model_id = 'stanford-crfm/BioMedLM'
# model_id = "codellama/CodeLlama-7b-hf"

In [ ]:
# Load the tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float32,
    device_map="auto"
)

tokenizer_config.json:   0%|          | 0.00/996 [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.80M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/4.54G [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/9.94G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

In [ ]:
retriever = db.as_retriever(search_kwargs={"k": 3}) # Retrieve top 3 docs

In [ ]:
pipe = pipeline(
 "text-generation",
 model=model,
 tokenizer=tokenizer,
 max_new_tokens=916,
 temperature=0.2,
 top_p=0.95,
 repetition_penalty=1.15
)

llm = HuggingFacePipeline(pipeline=pipe)

Device set to use cuda:0
<ipython-input-13-beb6535ba9b1>:11: LangChainDeprecationWarning: The class `HuggingFacePipeline` was deprecated in LangChain 0.0.37 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFacePipeline``.
  llm = HuggingFacePipeline(pipeline=pipe)


## Load in Patient Data

In [ ]:
df = pd.read_csv("/content/drive/MyDrive/cleaned_data.csv")

In [ ]:
def format_structured_patient_data(row):
    categories = {
        "Vital Signs": ["HR", "RR", "O2 sat"],
        "Blood Pressure": ["systolic arterial pressure", "Mean arterial pressure", "diastolic arterial pressure"],
        "Blood Gas": ["pH", "paO2", "Lactate", "BE", "central venous O2 saturation"],
        "Organ Function": ["NIRS", "Central Venous/ Right Atrial Pressure", "BUN", "Creatinine"]
    }

    prompt_parts = []
    for category, metrics in categories.items():
        category_data = []
        for metric in metrics:
            # Find columns matching this metric
            matching_cols = [col for col in row.index if metric in col and col != 'label_require_ECMO']

            if matching_cols:
                metric_values = []
                for col in matching_cols:
                    if pd.notna(row[col]):
                        level = col.split('_')[-1].strip() if '_' in col else ""
                        value = row[col]
                        metric_values.append(f"{level}: {value}")

                if metric_values:
                    category_data.append(f"- {metric}: {', '.join(metric_values)}")

        if category_data:
            prompt_parts.append(f"## {category}\n" + "\n".join(category_data))

    return "\n\n".join(prompt_parts)

## Methodology 1 of Prompting LLM





In [ ]:
template = """
You are an expert pediatric intensivist with deep knowledge of ECMO (Extracorporeal Membrane Oxygenation).

Here is data on the patient: {question}

Here is background information on ECMO: {context}

Again, please remember to start your answer with an explicit YES or NO recommendation on the use of ECMO.

Final Answer Here:
"""
# - List 3-5 specific data points supporting your decision
# - Include brief rationale

prompt = PromptTemplate(
    template=template,
    input_variables=["context", "question"]
)

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    return_source_documents=True,
    chain_type_kwargs={
        "prompt": prompt,
    }
)

In [ ]:
def ask_question(question):
    # Run the chain with the query parameter
    result = qa_chain({"query": question})

    return result

In [ ]:
data = format_structured_patient_data(df.iloc[2])

In [ ]:
response = ask_question(data)

/usr/local/lib/python3.11/dist-packages/transformers/generation/configuration_utils.py:628: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.2` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/transformers/generation/configuration_utils.py:633: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.95` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


In [ ]:
results = {}
for i in range(len(df)):
  data = format_structured_patient_data(df.iloc[i])
  response = ask_question(data)
  results[i] = {
      "result": response.get('result'),
      "source_documents": response.get('source_documents'),
      "actual_label": int(df.iloc[i]['label_require_ECMO'])
  }
  print(i)

/usr/local/lib/python3.11/dist-packages/transformers/generation/configuration_utils.py:628: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.2` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/transformers/generation/configuration_utils.py:633: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.95` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
/usr/local/lib/python3.11/dist-packages/transformers/generation/configuration_utils.py:628: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.2` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/pyth

0


/usr/local/lib/python3.11/dist-packages/transformers/generation/configuration_utils.py:628: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.2` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/transformers/generation/configuration_utils.py:633: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.95` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


1


/usr/local/lib/python3.11/dist-packages/transformers/generation/configuration_utils.py:628: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.2` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/transformers/generation/configuration_utils.py:633: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.95` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


...

16


/usr/local/lib/python3.11/dist-packages/transformers/generation/configuration_utils.py:628: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.2` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/transformers/generation/configuration_utils.py:633: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.95` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


17


/usr/local/lib/python3.11/dist-packages/transformers/generation/configuration_utils.py:628: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.2` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/transformers/generation/configuration_utils.py:633: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.95` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


18
19


In [ ]:
for i in range(len(df)):
  print(results[i]['result'][-25:])



Final Answer Here:

YES


Final Answer Here:

YES


Final Answer Here:

YES


Final Answer Here:

YES


Final Answer Here:

YES


Final Answer Here:

YES


Final Answer Here:

YES


Final Answer Here:

YES


Final Answer Here:

YES


Final Answer Here:

YES


Final Answer Here:

YES


Final Answer Here:

YES


Final Answer Here:

YES


Final Answer Here:

YES


Final Answer Here:

YES


Final Answer Here:

YES


Final Answer Here:

YES


Final Answer Here:

YES


Final Answer Here:

YES


Final Answer Here:

YES


## Methodology 2 of Prompting LLM

In [ ]:
template = """
You are an expert pediatric intensivist with deep knowledge of ECMO (Extracorporeal Membrane Oxygenation).

Read this patients data very carefully and summarize it in your response.

Here is data on the patient: {question}

Additionally, here is some information on ECMO: {context}

Please remember to provide a short insight into the patient's condition.
Use that info to make a clear, explicit YES or NO recommendation.

Summary and Answer Here:
"""
# - List 3-5 specific data points supporting your decision
# - Include brief rationale

prompt = PromptTemplate(
    template=template,
    input_variables=["context", "question"]
)

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    return_source_documents=True,
    chain_type_kwargs={
        "prompt": prompt,
    }
)

In [ ]:
#Example data pull
data = format_structured_patient_data(df.iloc[11])
response = ask_question(data)
print(response['result'][-300:])

<ipython-input-18-af6f610ab2b5>:3: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use :meth:`~invoke` instead.
  result = qa_chain({"query": question})
/usr/local/lib/python3.11/dist-packages/transformers/generation/configuration_utils.py:628: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.2` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/transformers/generation/configuration_utils.py:633: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.95` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


 to provide a short insight into the patient's condition.
Use that info to make a clear, explicit YES or NO recommendation.

Summary and Answer Here:

The patient has severe hypoxia, acidosis, and shock. The patient needs immediate support for life.

The patient should be placed on ECMO immediately.


In [ ]:
results_methodtwo = {}
for i in range(len(df)):
  data = format_structured_patient_data(df.iloc[i])
  response = ask_question(data)
  results_methodtwo[i] = {
      "result": response.get('result'),
      "source_documents": response.get('source_documents'),
      "actual_label": int(df.iloc[i]['label_require_ECMO'])
  }
  print(i)

/usr/local/lib/python3.11/dist-packages/transformers/generation/configuration_utils.py:628: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.2` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/transformers/generation/configuration_utils.py:633: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.95` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
/usr/local/lib/python3.11/dist-packages/transformers/generation/configuration_utils.py:628: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.2` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/pyth

0


/usr/local/lib/python3.11/dist-packages/transformers/generation/configuration_utils.py:628: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.2` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/transformers/generation/configuration_utils.py:633: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.95` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


1


/usr/local/lib/python3.11/dist-packages/transformers/generation/configuration_utils.py:628: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.2` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/transformers/generation/configuration_utils.py:633: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.95` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


...

16


/usr/local/lib/python3.11/dist-packages/transformers/generation/configuration_utils.py:628: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.2` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/transformers/generation/configuration_utils.py:633: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.95` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


17


/usr/local/lib/python3.11/dist-packages/transformers/generation/configuration_utils.py:628: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.2` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/transformers/generation/configuration_utils.py:633: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.95` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


18
19


In [ ]:
for i in range(len(df)):
  print(results_methodtwo[i]['result'][-400:])

45–68.7)
  pHh
7.28 (7.20–7.33)
7.30 (7.18–7.38)
  Mean airway pressure (mm Hg)i
17 (14–22.5)
22.3 (17.5–26.9)
Copyright © 2023 by the Society of Critical Care Medicine and the World Federation of Pediatric Intensive and Critical Care Societies.
Unauthorized reproduction of this article is prohibited
Bembea et al
366     
www.pccmjournal.org 
May 2023 • Volume 24 • Number 5
(37%) than pooled estimates of mortality reported in 
a recent systematic review and meta-analysis of stud-

Please remember to provide a short insight into the patient's condition.
Use that info to make a clear, explicit YES or NO recommendation.

Summary and Answer Here:

The patient has severe hypoxia, acidosis, and shock. The patient needs immediate ECMO support.

The patient should receive ECMO support immediately.
    
www.pccmjournal.org 
May 2023 • Volume 24 • Number 5
(37%) than pooled estimates of mortality reported in 
a recent systematic review and meta-analysis of stud-

Please remember to provide a sho

In [ ]:
print(results_methodtwo[i]['result'])


You are an expert pediatric intensivist with deep knowledge of ECMO (Extracorporeal Membrane Oxygenation).

Read this patients data very carefully and summarize it in your response.

Here is data on the patient: ## Vital Signs
- HR: low: 0.0, med: 100.0, high: 0.0
- RR: low: 5.0, med: 95.0, high: 0.0
- O2 sat: low: 2.0, med: 90.0, high: 8.0, low: 0.0, med: 90.0, high: 10.0

## Blood Pressure
- systolic arterial pressure: low: 0.0, med: 90.0, high: 10.0
- Mean arterial pressure: low: 0.0, med: 90.0, high: 10.0
- diastolic arterial pressure: low: 0.0, med: 90.0, high: 10.0

## Blood Gas
- pH: low: 0.0, med: 80.0, high: 20.0
- paO2: low: 0.0, med: 90.0, high: 10.0
- Lactate: low: 90.0, med: 10.0, high: 0.0
- BE: low: 95.0, med: 5.0, high: 0.0
- central venous O2 saturation: low: 0.0, med: 90.0, high: 10.0

## Organ Function
- NIRS: low: 0.0, med: 30.0, high: 70.0
- Central Venous/ Right Atrial Pressure: low: 0.0, med: 98.0, high: 2.0
- BUN: low: 1.0, med: 99.0, high: 0.0
- Creatinine: lo